# Notebook_Proceso_PLN2

**Objetivo:** reproducir el proceso 2 del TFM.

Proceso documentado:

`Base_Datos_Original_Anonimizada_Procesada_TFM.xlsx` → reconstrucción de completitud + segmentación estructural → `Base_Ordenada_Subsets_TFM.xlsx`

El proceso original no estaba documentado. Este cuaderno reconstruye el resultado mediante inferencia reproducible:

1. `TOTAL_RESULTADOS` se calcula como conteo de completitud sobre 31 campos clínicos, temporales y normalizados.
2. `PORCENTAJE_COMPLETITUD` se calcula como `TOTAL_RESULTADOS / 31 * 100`.
3. `SUBSET_EXAMENES` se reproduce con `KMeans(n_clusters=4, random_state=42, n_init=10)` aplicado a la matriz de presencia de las variables clínicas normalizadas.
4. Se exportan las hojas `BASE_COMPLETA`, `BATERIA_A`, `BATERIA_B`, `BATERIA_C` y `BATERIA_D`.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


In [ ]:
INPUT_XLSX = Path("Base_Datos_Original_Anonimizada_Procesada_TFM.xlsx")
OUTPUT_XLSX = Path("Base_Ordenada_Subsets_TFM.xlsx")

# Referencia opcional para validación, si se desea comparar contra un archivo previamente generado
REFERENCE_XLSX = Path("Base_Ordenada_Subsets_TFM_REFERENCIA.xlsx")

print("Entrada:", INPUT_XLSX.resolve())
print("Salida:", OUTPUT_XLSX.resolve())


In [ ]:
df = pd.read_excel(INPUT_XLSX, sheet_name="Cohorte_Anonimizada")
df.columns = [str(c).strip() for c in df.columns]

print("Filas:", len(df))
print("Columnas:", df.shape[1])
df.head()


In [ ]:
# Campos utilizados para TOTAL_RESULTADOS.
# La suma de estos campos produce un máximo de 31 resultados por registro.
DATE_COLUMNS = [
    "UltimaAtencion_Anio",
    "UltimaAtencion_Mes",
    "DiasDesdePrimeraAtencion",
]

ORIGINAL_CLINICAL_COLUMNS = [
    "Edad",
    "Sexo",
    "Peso",
    "Altura",
    "IMC",
    "PA_Sistolica",
    "PA_Diastolica",
    "Glicemia",
    "ColesterolTotal",
    "HDL",
    "LDL",
    "Trigliceridos",
    "Hemoglobina",
    "Creatinina",
    "Tabaquismo",
    "Diabetes",
]

NORMALIZED_CLINICAL_COLUMNS = [
    "Peso_kg",
    "Altura_m",
    "IMC_num",
    "PA_Sistolica_mmHg",
    "PA_Diastolica_mmHg",
    "Glicemia_mg_dl",
    "ColesterolTotal_mg_dl",
    "HDL_mg_dl",
    "LDL_mg_dl",
    "Trigliceridos_mg_dl",
    "Hemoglobina_gr_pct",
    "Creatinina_mg_dl",
]

COMPLETENESS_COLUMNS = [c for c in DATE_COLUMNS + ORIGINAL_CLINICAL_COLUMNS + NORMALIZED_CLINICAL_COLUMNS if c in df.columns]
CLUSTER_COLUMNS = [c for c in NORMALIZED_CLINICAL_COLUMNS if c in df.columns]

assert len(COMPLETENESS_COLUMNS) == 31, f"Se esperaban 31 columnas de completitud y se encontraron {len(COMPLETENESS_COLUMNS)}."
assert len(CLUSTER_COLUMNS) == 12, f"Se esperaban 12 columnas normalizadas para clustering y se encontraron {len(CLUSTER_COLUMNS)}."

print("Columnas de completitud:", len(COMPLETENESS_COLUMNS))
print("Columnas de clustering:", len(CLUSTER_COLUMNS))


In [ ]:
df_work = df.copy()

# Indicadores de completitud
numerator = df_work[COMPLETENESS_COLUMNS].notna().sum(axis=1)
df_work["TOTAL_RESULTADOS"] = numerator.astype(int)
df_work["PORCENTAJE_COMPLETITUD"] = ((numerator / len(COMPLETENESS_COLUMNS)) * 100).round(2)

# Segmentación estructural mediante matriz de presencia/ausencia.
presence_matrix = df_work[CLUSTER_COLUMNS].notna().astype(int)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters = kmeans.fit_predict(presence_matrix)
df_work["BATERIA_CLUSTER"] = clusters

# Mapeo reproducido desde la estructura resultante del clustering.
# Con random_state=42, los clusters quedan asociados a estas baterías.
cluster_map = {
    0: "BATERIA_A",
    1: "BATERIA_B",
    2: "BATERIA_C",
    3: "BATERIA_D",
}

df_work["SUBSET_EXAMENES"] = df_work["BATERIA_CLUSTER"].map(cluster_map)
df_work = df_work.drop(columns=["BATERIA_CLUSTER"])

# Ordenamiento reproducido: mayor completitud primero; luego orden alfabético de batería; luego PACIENTE_ID.
df_work = df_work.sort_values(
    by=["TOTAL_RESULTADOS", "SUBSET_EXAMENES", "PACIENTE_ID"],
    ascending=[False, True, True],
).reset_index(drop=True)

resumen = df_work[["SUBSET_EXAMENES", "TOTAL_RESULTADOS", "PORCENTAJE_COMPLETITUD"]].value_counts().reset_index(name="N")
resumen = resumen.sort_values(["SUBSET_EXAMENES", "TOTAL_RESULTADOS"], ascending=[True, False])
resumen


In [ ]:
sheets = {"BASE_COMPLETA": df_work}
for subset_name in ["BATERIA_A", "BATERIA_B", "BATERIA_C", "BATERIA_D"]:
    subset_df = df_work[df_work["SUBSET_EXAMENES"] == subset_name].copy().reset_index(drop=True)
    sheets[subset_name] = subset_df

with pd.ExcelWriter(OUTPUT_XLSX, engine="xlsxwriter") as writer:
    for sheet_name, data in sheets.items():
        data.to_excel(writer, sheet_name=sheet_name, index=False)

print("Archivo generado:", OUTPUT_XLSX.resolve())
for sheet_name, data in sheets.items():
    print(f"{sheet_name}: {data.shape[0]} filas, {data.shape[1]} columnas")


In [ ]:
# Validación de consistencia interna
assert sheets["BASE_COMPLETA"].shape[0] == sum(sheets[s].shape[0] for s in ["BATERIA_A", "BATERIA_B", "BATERIA_C", "BATERIA_D"]), "Las hojas de baterías no suman la base completa."
assert sheets["BASE_COMPLETA"]["PACIENTE_ID"].is_unique, "PACIENTE_ID debe ser único en BASE_COMPLETA."
assert set(sheets["BASE_COMPLETA"]["SUBSET_EXAMENES"].unique()) == {"BATERIA_A", "BATERIA_B", "BATERIA_C", "BATERIA_D"}, "Subconjuntos inesperados."

print("Validación interna: OK")
print(sheets["BASE_COMPLETA"][["PACIENTE_ID", "TOTAL_RESULTADOS", "PORCENTAJE_COMPLETITUD", "SUBSET_EXAMENES"]].head(10))
print("\nConteo por batería:")
print(sheets["BASE_COMPLETA"]["SUBSET_EXAMENES"].value_counts().sort_index())


In [ ]:
# Validación opcional contra archivo de referencia.
# Para usarla, coloque la referencia con el nombre Base_Ordenada_Subsets_TFM_REFERENCIA.xlsx en el mismo directorio.
if REFERENCE_XLSX.exists():
    ref = pd.read_excel(REFERENCE_XLSX, sheet_name="BASE_COMPLETA")
    generated = sheets["BASE_COMPLETA"]
    common_cols = [c for c in ref.columns if c in generated.columns]
    same_shape = ref.shape == generated.shape
    same_common_values = ref[common_cols].reset_index(drop=True).equals(generated[common_cols].reset_index(drop=True))
    print("Misma forma:", same_shape)
    print("Mismos valores en columnas comunes:", same_common_values)
else:
    print("No se encontró archivo de referencia; se omitió validación externa.")
